# 📓 03 — GR00T Policy Deep DiveNVIDIA GR00T (Generalist Robot 00 Technology) is a foundation model for robot control.`strands-robots` provides `Gr00tPolicy` with two modes:- **Service mode**: Connects to a GR00T inference server via ZMQ- **Local mode**: Loads the model directly (requires Isaac-GR00T package)

## Data ConfigurationsGR00T uses typed "data configs" that define how robot observations map to model modalities.25 configs are included in `data_configs.json`, organized with `_extends` inheritance.

In [ ]:
from strands_robots.policies.groot.data_config import (    Gr00tDataConfig, DATA_CONFIG_MAP, load_data_config, create_custom_data_config)# List all available configsprint(f"Available configs ({len(DATA_CONFIG_MAP)}):")for name in sorted(DATA_CONFIG_MAP.keys()):    cfg = DATA_CONFIG_MAP[name]    print(f"  {name:30s} videos={cfg.video_keys}  state={cfg.state_keys}")

In [ ]:
# Load and inspect a specific configcfg = load_data_config("so100_dualcam")print(f"Config: {cfg.name}")print(f"  Video keys:   {cfg.video_keys}")print(f"  State keys:   {cfg.state_keys}")print(f"  Action keys:  {cfg.action_keys}")print(f"  Language keys: {cfg.language_keys}")print(f"  Obs indices:  {cfg.observation_indices}")print(f"  Action horizon: {cfg.action_indices}")# Get the modality_config dict (used by Isaac-GR00T loaders)mc = cfg.modality_config()print(f"\nModality config keys: {list(mc.keys())}")for k, v in mc.items():    print(f"  {k}: modality_keys={v.modality_keys}, delta_indices={v.delta_indices}")

## Custom Data ConfigsCreate configs for new robots at runtime — they're immediately usable with `load_data_config()`.

In [ ]:
custom = create_custom_data_config(    name="my_robot_3cam",    video_keys=["video.left", "video.right", "video.overhead"],    state_keys=["state.arm", "state.gripper"],    action_keys=["action.arm", "action.gripper"],)print(f"Custom config: {custom.name}")print(f"  Videos: {custom.video_keys}")# Verify it's registeredloaded = load_data_config("my_robot_3cam")print(f"  Loaded back: {loaded.name} ✅")

## Observation & Action MappingsGR00T's model uses its own internal key names. The mapping system translates betweenrobot sensor names and model modality keys — no positional guessing.```python# Explicit mapping (recommended for custom robots)policy = Gr00tPolicy(    data_config="so100_dualcam",    model_path="nvidia/GR00T-N1.6-3B",    observation_mapping={        "front_cam": "video.front",     # robot key → model key        "wrist_cam": "video.wrist",        "joint_pos": "state.single_arm",        "grip_pos":  "state.gripper",    },    action_mapping={        "action.single_arm": "joint_pos",  # model key → robot key        "action.gripper": "grip_pos",    },)```Auto-inference also works: exact name match first, then positional fallback.

## ZMQ Inference ClientThe `Gr00tInferenceClient` handles communication with Isaac-GR00T Docker containers.

In [ ]:
from strands_robots.policies.groot.client import Gr00tInferenceClient, MsgSerializerimport numpy as np# MsgSerializer handles numpy arrays + ModalityConfig over msgpackdata = {    "test_array": np.random.rand(3, 3).astype(np.float32),    "test_string": "hello",}# Round-trip: dict → bytes → dictencoded = MsgSerializer.to_bytes(data)decoded = MsgSerializer.from_bytes(encoded)print(f"Encoded size: {len(encoded)} bytes")print(f"Array preserved: {np.allclose(data['test_array'], decoded['test_array'])}")print(f"String preserved: {data['test_string'] == decoded['test_string']}")

## GR00T Inference Tool (Docker Management)The `gr00t_inference` Strands tool manages Docker containers running GR00T services.```pythonfrom strands_robots import gr00t_inferencefrom strands import Agentagent = Agent(tools=[gr00t_inference])# Start inference serviceagent.tool.gr00t_inference(    action="start",    checkpoint_path="/data/checkpoints/model",    port=8000,    data_config="so100_dualcam",    http_server=True,  # REST API mode)# Check statusagent.tool.gr00t_inference(action="status", port=8000)# With TensorRT accelerationagent.tool.gr00t_inference(    action="start",    checkpoint_path="/data/checkpoints/model",    port=5555,    use_tensorrt=True,    vit_dtype="fp8",    llm_dtype="nvfp4",    dit_dtype="fp8",)```